# Major and Minor Player with Partial Observations

The major-minor linear-quadratic game of Firoozi and Caines (2020), with one minor player. A major player (for
example a large institution) and a minor player each steer a state, at a quadratic cost. The minor player's state
is pulled by the major player's state, and the minor player wants to stay close to a multiple of it.

## 1. The game

The states $x^0$ (major) and $x^1$ (minor) follow

$$
\begin{aligned}
dx^0_t &= (A_0\, x^0_t + B_0\, u^0_t)\,dt + D_0\,dw^0_t, \\[4pt]
dx^1_t &= (A\, x^1_t + G\, x^0_t + B\, u^1_t)\,dt + D\,dw^1_t .
\end{aligned}
$$

Neither player observes its state directly. The major player observes $y^0$, a noisy measurement of its own state.
The minor player observes $y^1$, noisy measurements of its own state and of the major player's state:

$$
\begin{aligned}
dy^0_t &= L_0\, x^0_t\,dt + \sigma_{v_0}\,dv^0_t, \\[4pt]
dy^1_t &= (L_1\, x^1_t,\; L_2\, x^0_t)\,dt + \sigma_v\,dv^1_t .
\end{aligned}
$$

Neither player observes the other's observation path, so neither player's information contains the other's. The
major player steers its state toward $\eta_0$, and the minor player steers its state toward $H_1 x^0 + \eta$. They
minimize the discounted costs

$$
\begin{aligned}
J^0 &= \mathbb{E} \int_0^T e^{-\rho t} \big[ Q_0\,(x^0_t - \eta_0)^2 + R_0\,(u^0_t)^2 \big]\,dt, \\[4pt]
J^1 &= \mathbb{E} \int_0^T e^{-\rho t} \big[ Q\,(x^1_t - H_1 x^0_t - \eta)^2 + R\,(u^1_t)^2 \big]\,dt .
\end{aligned}
$$

## 2. The exact solution

The game is linear-quadratic and Gaussian, so each player's optimal control is linear in its estimate of the state
it needs.

- The major player needs only its own state. It estimates $x^0$ from $y^0$ by a Kalman filter, $\hat x^0_t$, and its
  control is linear in $\hat x^0_t$.
- The minor player needs $x^1$ and $x^0$, and also the major player's estimate $\hat x^0_t$, because the major
  player's control, and hence $x^0$, depends on $\hat x^0_t$. The minor player therefore filters the three
  quantities $(x^1, x^0, \hat x^0)$ from $y^1$: it estimates the major player's estimate. Its control is linear in
  that filter.

Both filters and both controls are computed exactly for the discretised game, by the discrete Kalman filter and a
discounted backward Riccati recursion (`MajorMinor.exact`). The learned method computes none of this: each player's
network takes only its own observation path as input.

## 3. Differences from the paper

- One minor player instead of $N$ minor players coupled through their average; the terms of the costs that depend
  on that average are zero.
- A finite horizon $T = 5$ with discounting, instead of an infinite horizon.
- Scalar states, with the coefficients $L_0, L_1, L_2$ in place of the paper's observation matrices.

In [ ]:
import sys, numpy as np, torch, matplotlib.pyplot as plt
sys.path.insert(0, '../src')
from partial_info_games import Solver, first_order_residual, relative_l2, use_device, validation_paths
from partial_info_games import use_style, plot_controls
use_style()
DEVICE = 'cpu'                     # 'mps' (Apple GPU) or 'cuda' to use a GPU
device = use_device(DEVICE)
to_players = lambda controls: [torch.tensor(controls[:, i], dtype=torch.float32) for i in range(2)]
print('device:', device)

from partial_info_games.games import MajorMinor

## 4. Learned and exact equilibria

The learned method at the small budget of `two_traders.ipynb` (5,000 paths, 6 iterations), so that the notebook runs
in a few minutes on a CPU.

In [ ]:
game = MajorMinor()
solver = Solver(game, damping=0.7, steps=300, batch=1000, n_paths=5000)
exact = to_players(game.exact(solver.validation))
learned, _ = solver.run(6, average_from=3, reference=exact)
print(f'relative L2 error: major {relative_l2(learned[0], exact[0]):.3f}, minor {relative_l2(learned[1], exact[1]):.3f}')
plot_controls(learned, exact, game.dt, ['major player', 'minor player'])